# Baseline model

Load the processed dataset from Google Drive, split by airfoil geometry, train the project's linear baseline, and evaluate it on unseen geometries.

Use a Colab CPU runtime and run the cells in order. Push local code changes before running: cloning reads GitHub, not your local working tree. The setup cell leaves existing checkouts untouched; use a fresh runtime to pick up a newly pushed version.

The processed CSV must already be in Drive. Each run saves its model, per-target metrics, plot, and reproducibility metadata in a new Drive folder under `airfoil-ml-surrogate/experiments/baseline/`.

In [ ]:
from pathlib import Path
import subprocess
import sys

from google.colab import drive

PROJECT_ROOT = Path('/content/airfoil-ml-surrogate')
REPO_URL = 'https://github.com/gaurav-zz/ML-for-CFD.git'
if not PROJECT_ROOT.exists():
    subprocess.run(['git', 'clone', REPO_URL, str(PROJECT_ROOT)], check=True)
if not (PROJECT_ROOT / '.git').exists() or not (PROJECT_ROOT / 'requirements.txt').is_file():
    raise FileNotFoundError(f'Expected the project Git checkout at {PROJECT_ROOT}. Use a fresh Colab runtime.')
subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '-q', '-r', str(PROJECT_ROOT / 'requirements.txt')],
    check=True,
)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
print(f'Source checkout: {PROJECT_ROOT}')

In [ ]:
from datetime import datetime, timezone
from importlib.metadata import version
import hashlib
import json

import pandas as pd

drive.mount('/content/drive')

DRIVE_ROOT = Path('/content/drive/MyDrive/airfoil-ml-surrogate')
PROCESSED_PATH = DRIVE_ROOT / 'data' / 'processed' / 'airfoil_learning.csv'

if not PROCESSED_PATH.exists():
    raise FileNotFoundError(f'Processed dataset not found: {PROCESSED_PATH}')

from src.data.preprocessing import geometry_aware_split
from src.data.validation import validate_dataframe
from src.training.evaluate import evaluate_model
from src.training.train import train_model

df = pd.read_csv(PROCESSED_PATH)
validate_dataframe(df, required_columns=['geometry_id', 'Alpha', 'Re', 'Cl', 'Cd'])

# Fingerprint the input without holding a second copy of the CSV in memory.
dataset_digest = hashlib.sha256()
with PROCESSED_PATH.open('rb') as data_file:
    for chunk in iter(lambda: data_file.read(1024 * 1024), b''):
        dataset_digest.update(chunk)

print(f'Loaded dataset with {len(df):,} rows and {len(df.columns):,} columns.')
print(f'Unique geometries: {df["geometry_id"].nunique():,}')
df.head()

## Prepare features and targets

Geometry features are combined with the operating conditions. `geometry_id` is used only for grouping and is not passed to the model.

In [ ]:
geometry_columns = sorted(column for column in df.columns if column.startswith('geom_'))
if not geometry_columns:
    raise ValueError('No geom_ geometry features found in the processed dataset.')
feature_columns = geometry_columns + ['Alpha', 'Re']
if 'Ncrit' in df.columns:
    feature_columns.append('Ncrit')
target_columns = ['Cl', 'Cd']

split_parameters = {'seed': 42, 'train_size': 0.70, 'validation_size': 0.15}
split = geometry_aware_split(
    frame=df,
    groups=df['geometry_id'],
    **split_parameters,
)

X_train = split.train[feature_columns]
X_validation = split.validation[feature_columns]
X_test = split.test[feature_columns]
y_train = split.train[target_columns]
y_validation = split.validation[target_columns]
y_test = split.test[target_columns]

train_geometries = set(split.train['geometry_id'])
validation_geometries = set(split.validation['geometry_id'])
test_geometries = set(split.test['geometry_id'])
assert train_geometries.isdisjoint(validation_geometries)
assert train_geometries.isdisjoint(test_geometries)
assert validation_geometries.isdisjoint(test_geometries)

print(f'Features: {len(feature_columns)}')
print(f'Train: {len(X_train):,} rows, {len(train_geometries):,} geometries')
print(f'Validation: {len(X_validation):,} rows, {len(validation_geometries):,} geometries')
print(f'Test: {len(X_test):,} rows, {len(test_geometries):,} geometries')
print('Geometry leakage check passed.')

## Train the linear baseline

The project linear model contains its own `StandardScaler`. It is fitted on `X_train` only, so no separate scaling step is needed here. A new timestamped output directory keeps earlier runs intact.

In [ ]:
run_started = datetime.now(timezone.utc)
RUN_OUTPUT = DRIVE_ROOT / 'experiments' / 'baseline' / run_started.strftime('%Y%m%dT%H%M%S%fZ')
RUN_OUTPUT.mkdir(parents=True, exist_ok=False)

model, training_time = train_model(
    name='linear',
    x_train=X_train,
    y_train=y_train,
    output_dir=RUN_OUTPUT,
)

run_metadata = {
    'created_at_utc': run_started.isoformat(),
    'model': 'linear',
    'training_time_seconds': training_time,
    'git_commit': subprocess.check_output(['git', 'rev-parse', 'HEAD'], cwd=PROJECT_ROOT, text=True).strip(),
    'git_dirty': bool(subprocess.check_output(['git', 'status', '--porcelain'], cwd=PROJECT_ROOT, text=True).strip()),
    'python_version': sys.version,
    'packages': {name: version(name) for name in ['numpy', 'pandas', 'scikit-learn', 'joblib', 'matplotlib']},
    'dataset': {
        'path': str(PROCESSED_PATH),
        'sha256': dataset_digest.hexdigest(),
        'rows': len(df),
        'columns': len(df.columns),
    },
    'feature_columns': feature_columns,
    'target_columns': target_columns,
    'split_parameters': split_parameters,
    'split_geometries': {
        'train': sorted(train_geometries),
        'validation': sorted(validation_geometries),
        'test': sorted(test_geometries),
    },
    'split_rows': {'train': len(X_train), 'validation': len(X_validation), 'test': len(X_test)},
}
if run_metadata['git_dirty']:
    print('Warning: the source checkout has uncommitted changes; the commit ID alone cannot reproduce this run.')

print(f'Training time: {training_time:.3f} seconds')
print(f'Model saved to: {RUN_OUTPUT / "linear.joblib"}')

## Evaluate and save results

Report MAE, RMSE, and R-squared separately for `Cl` and `Cd`; their different scales must not be averaged together. R-squared is undefined for a one-row holdout and is saved as JSON `null` (blank in CSV). Inference timing covers the full batch of both targets. Use validation scores for model decisions and reserve test scores for final reporting.

In [ ]:
evaluations = {
    'validation': evaluate_model(model, X_validation, y_validation, target_names=target_columns),
    'test': evaluate_model(model, X_test, y_test, target_names=target_columns),
}

metrics = pd.DataFrame([
    {'split': partition, 'target': target, **scores}
    for partition, report in evaluations.items()
    for target, scores in report['targets'].items()
]).set_index(['split', 'target'])
metrics.to_csv(RUN_OUTPUT / 'metrics.csv')
run_metadata['evaluation'] = evaluations
(RUN_OUTPUT / 'run.json').write_text(json.dumps(run_metadata, indent=2, allow_nan=False) + '\n', encoding='utf-8')
display(metrics)
display(pd.DataFrame([
    {'split': partition, 'inference_time_seconds': report['inference_time_seconds'], 'n_predictions': report['n_predictions']}
    for partition, report in evaluations.items()
]).set_index('split'))
print(f'Metrics and run metadata saved to: {RUN_OUTPUT}')

In [ ]:
import matplotlib.pyplot as plt

predictions = model.predict(X_test)
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

for index, target in enumerate(target_columns):
    actual = y_test[target].to_numpy()
    predicted = predictions[:, index]
    axes[index].scatter(actual, predicted, alpha=0.25)
    lower = min(actual.min(), predicted.min())
    upper = max(actual.max(), predicted.max())
    axes[index].plot([lower, upper], [lower, upper], '--', color='black')
    axes[index].set_title(f'{target}: predicted vs actual')
    axes[index].set_xlabel(f'Actual {target}')
    axes[index].set_ylabel(f'Predicted {target}')
    axes[index].grid(alpha=0.3)

plt.tight_layout()
fig.savefig(RUN_OUTPUT / 'predicted_vs_actual.png', dpi=150, bbox_inches='tight')
plt.show()
plt.close(fig)
print(f'Plot saved to: {RUN_OUTPUT / "predicted_vs_actual.png"}')